# WHW10 starter notebook: one double integral, three ways

We're going to evaluate

$$I = \iint_R xy \, dA, \qquad R = \text{the quarter disk } x^2 + y^2 \le 1,\ x \ge 0,\ y \ge 0,$$

three ways:

1. **by hand** in polar coordinates (on paper);
2. **with SymPy**;
3. **with a NumPy Riemann sum**: chop the square $[0, 1] \times [0, 1]$ into an
   $N \times N$ grid of little square cells, sample $f$ at the center of each
   cell, and zero out every sample point that falls outside the quarter disk.

Then we'll ask how fast the Riemann sum closes in on the exact answer as $N$
grows. Here's the problem:

- **Before running the Riemann sum**, predict two things and write them down:
  will it come out above or below the exact answer, and if you double $N$, by
  what factor should the error shrink?
- Run the Riemann sum for every $N$ from 10 to 320 in steps of 5 (that includes
  $N = 10, 20, 40, 80, 160$), plot the absolute error against $N$ on log-log
  axes, and estimate the slope by comparing with reference lines of slope $-1$
  and $-2$.
- Explain in two or three sentences why the observed slope is what it is,
  rather than the slope you get integrating a smooth function over a plain
  rectangle (section 6 does that one for you).
- Report one thing that went wrong (mismatched `meshgrid` indexing is the classic).

We have *not* done double integrals in SymPy, grids in NumPy, or "is this point
inside the region?" tests yet, so this notebook shows you each one on a
*different* example first, then leaves the quarter disk to you.

## 0. Setup

Same setup as always: matplotlib for plotting, sympy for symbolic manipulation,
and numpy for numerical python calculations.

In [ ]:
%matplotlib inline
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from sympy import *
init_printing()

## 1. Predict first (before you run anything)

Picture the grid of cells laid over the quarter disk, with each cell counted
in or out according to whether its center is inside.

1. Will the Riemann sum come out **above** or **below** the exact answer? Why?
2. If you double $N$, by what factor should the error shrink?

Double-click the cell below, type your prediction, and press Shift-Enter.

*(your prediction here)*

## 2. By hand, in polar coordinates

Do this one on paper: set up $I$ in polar coordinates (don't forget the extra
factor of $r$ in $dA = r\,dr\,d\theta$) and evaluate it. Write your answer here:

*(your answer here)*

## 3. Double integrals in SymPy

`integrate` takes one tuple per variable: `(variable, lower, upper)`. With two
tuples you get a double integral. The **first** tuple is the **inside** integral,
so its limits are allowed to depend on the outer variable.

Demo: integrate $x + y$ over the triangle $0 \le y \le x$, $0 \le x \le 1$. For each
$x$, $y$ runs from 0 up to the line $y = x$:

$$\int_0^1 \int_0^x (x + y)\, dy\, dx .$$

In [ ]:
x, y = sp.symbols('x y')
triangle_answer = integrate(x + y, (y, 0, x), (x, 0, 1))
triangle_answer

- `(y, 0, x)` comes first, so SymPy does the $y$ integral first, from $y = 0$ to $y = x$.
- `(x, 0, 1)` comes second: the outside integral, with plain numbers for limits.
- Swap the order of the tuples and you'd be asking for $\int_0^x \int_0^1 \dots\, dx\, dy$,
  which leaves an $x$ in the answer. If SymPy hands you something with a leftover
  variable in it, that's usually why.

Polar coordinates work the same way; you just have to put in the $r$ from
$dA = r\,dr\,d\theta$ yourself. Here's the area of a disk of radius 2:

In [ ]:
r, theta = sp.symbols('r theta', positive=True)
disk_area = integrate(1 * r, (r, 0, 2), (theta, 0, 2*pi))
disk_area

**Your turn.** Change the line below to compute our $I$ (either Cartesian or
polar is fine; the quarter disk's upper edge is $y = \sqrt{1 - x^2}$). Keep the
name `exact`: section 9 uses it.

In [ ]:
# Change the line below.
exact = integrate(x + y, (y, 0, x), (x, 0, 1))
exact

## 4. A one-dimensional Riemann sum, and reading a slope off a log-log plot

Before we go to two dimensions, here's the one-dimensional version, for a
smooth function on a plain interval, where we know the answer:

$$\int_0^{\pi/2} \cos x \, dx = 1 .$$

Chop $[0, \pi/2]$ into $N$ pieces of width $h = (\pi/2)/N$ and sample at the
**middle** of each piece:

- `np.arange(N)` is the array `[0, 1, 2, ..., N-1]`.
- `(np.arange(N) + 0.5) * h` is then the array of midpoints, $h/2, 3h/2, \dots$
- `np.sum(...)` adds up every entry of an array. Times $h$, that's the Riemann sum.

In [ ]:
def riemann_1d_demo(N):
    h = (np.pi / 2) / N
    x_mid = (np.arange(N) + 0.5) * h
    return np.sum(np.cos(x_mid)) * h

print(riemann_1d_demo(10), riemann_1d_demo(20))

Now let's see how the error shrinks with $N$. As in the WHW06 starter: if
error $\approx C N^p$, then $\log(\text{error}) = \log C + p \log N$, so on log-log
axes it's a straight line with slope $p$. In the WHW06 starter we measured $p$
as rise over run between two points, which works great on a clean straight
line. When the data are ragged, two points can tell you almost anything, so
here I like to draw reference lines of known slope next to the data and see
which one it runs parallel to.
The helper below draws lines of slope $-1$ and $-2$ through the first data point.

In [ ]:
def add_reference_lines(N_vals, error_vals):
    plt.loglog(N_vals, error_vals[0] * (N_vals / N_vals[0])**-1.0, 'k:', label='slope -1')
    plt.loglog(N_vals, error_vals[0] * (N_vals / N_vals[0])**-2.0, 'k--', label='slope -2')

N_demo = np.array([10, 20, 40, 80, 160, 320])
errors_1d = np.zeros(len(N_demo))
for i in range(len(N_demo)):
    errors_1d[i] = abs(riemann_1d_demo(N_demo[i]) - 1.0)

plt.loglog(N_demo, errors_1d, 'o-', label='1D midpoint sum of cos x')
add_reference_lines(N_demo, errors_1d)
plt.xlabel('N')
plt.ylabel('absolute error')
plt.legend()
plt.show()
print('error ratio each time N doubles:', errors_1d[:-1] / errors_1d[1:])

The data run parallel to the slope $-2$ line: double $N$ and the error drops by
a factor of 4. That's what midpoint sums do for a smooth function.

## 5. Grids in two dimensions: `np.meshgrid`

For a double integral we need every *pair* $(x, y)$ on the grid. `np.meshgrid`
takes a 1D array of $x$ values and a 1D array of $y$ values and hands back two 2D
arrays, `X` and `Y`, with one entry per grid point. Let's look at a tiny one,
with 3 $x$ values and 2 $y$ values, so we can see which way is which:

In [ ]:
x_small = np.array([0, 1, 2])
y_small = np.array([10, 20])
X_small, Y_small = np.meshgrid(x_small, y_small, indexing='xy')
print('X =')
print(X_small)
print('Y =')
print(Y_small)
print('shape:', X_small.shape)

Here's the thing to remember, since it's the classic way to get this wrong:

- With `indexing='xy'`, the arrays have **one row per $y$ value and one column
  per $x$ value**: shape `(len(y), len(x))`, here `(2, 3)`.
- `X` changes as you go **across a row** (left to right); it's the same all
  the way down a column.
- `Y` changes as you go **down a column**; it's the same all the way across a row.
- `X_small[i, j]` is the point $(x_j, y_i)$. Note the order: row index first, and the row index goes with $y$.
- With `indexing='ij'` it's the other way around: shape `(len(x), len(y))`, and
  `X` changes down a column. That's also fine, as long as you know which one you
  have. Mixing them up is the classic bug. In this notebook we always say
  `indexing='xy'` out loud so there's no guessing.

Once we have `X` and `Y`, any formula in them is computed at every grid point at
once (element-wise, as in the WHW06 starter): `X * Y` is the 2D array of $xy$ values.

## 6. A two-dimensional Riemann sum on a plain rectangle

Demo: a smooth function on a plain square, where we know the answer:

$$\int_0^{\pi/2}\!\!\int_0^{\pi/2} \cos x \cos y \, dy\, dx = 1 .$$

Same midpoints as section 4, now in both directions. Each cell has area $h^2$.

In [ ]:
def riemann_rectangle_demo(N):
    h = (np.pi / 2) / N
    x_mid = (np.arange(N) + 0.5) * h
    y_mid = (np.arange(N) + 0.5) * h
    X, Y = np.meshgrid(x_mid, y_mid, indexing='xy')
    f_vals = np.cos(X) * np.cos(Y)
    return np.sum(f_vals) * h * h

errors_rect = np.zeros(len(N_demo))
for i in range(len(N_demo)):
    errors_rect[i] = abs(riemann_rectangle_demo(N_demo[i]) - 1.0)

plt.loglog(N_demo, errors_rect, 'o-', label='2D midpoint sum, smooth f, plain rectangle')
add_reference_lines(N_demo, errors_rect)
plt.xlabel('N')
plt.ylabel('absolute error')
plt.legend()
plt.show()
print('error ratio each time N doubles:', errors_rect[:-1] / errors_rect[1:])

Slope $-2$ again: a smooth function on a region that lines up with the grid.
**That's the "plain rectangle" slope the problem asks you to compare against.**

## 7. Points inside a region: boolean masks

Now a region that is *not* a rectangle. Demo: the area of the full unit disk,
$x^2 + y^2 \le 1$, which is $\pi$. Put a grid over the square $[-1, 1] \times [-1, 1]$
and ask, at every grid point, "is this point inside the disk?"

- `X**2 + Y**2 <= 1` is a 2D array of `True` / `False`, one per grid point. That's
  called a **boolean mask**.
- In arithmetic, `True` counts as 1 and `False` as 0. So `f_vals * inside` keeps
  $f$ where the point is inside and **zeros it out** where it isn't, and
  `np.sum(inside)` counts the points inside.
- For the area, $f = 1$, so the area is (number of cells inside) $\times\, h^2$.

In [ ]:
def disk_area_demo(N):
    h = 2.0 / N
    x_mid = -1 + (np.arange(N) + 0.5) * h
    y_mid = -1 + (np.arange(N) + 0.5) * h
    X, Y = np.meshgrid(x_mid, y_mid, indexing='xy')
    inside = X**2 + Y**2 <= 1
    return np.sum(inside) * h * h, X, Y, inside

area_20, X_20, Y_20, inside_20 = disk_area_demo(20)
print('N = 20: area estimate', area_20, ' exact', np.pi, ' error', area_20 - np.pi)

Here's what that grid "sees". `X_20[inside_20]` picks out just the entries where the
mask is `True` (and `~inside_20` means "not inside"). Each dot is the center of one
cell, and each cell is counted all-in or all-out.

In [ ]:
angles = np.linspace(0, 2 * np.pi, 400)
plt.figure(figsize=(5, 5))
plt.plot(X_20[inside_20], Y_20[inside_20], 'o', color='tab:blue', label='center inside: whole cell counted')
plt.plot(X_20[~inside_20], Y_20[~inside_20], 'o', color='lightgray', label='center outside: cell dropped')
plt.plot(np.cos(angles), np.sin(angles), 'k', label='the real edge')
plt.gca().set_aspect('equal')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.08))
plt.show()

## 8. Why a curved edge changes the story

In the rectangle demo, every cell is completely inside the region. The only
error is that $f$ isn't constant across a cell, and sampling at the middle of
each cell makes that error small: slope $-2$.

With a curved edge there's a second, different kind of error. Look at the
picture: if you colored in every blue cell, the region you'd get has a
**staircase** edge, not a round one. A cell the circle cuts through is counted
completely in or completely out, depending only on which side its center lands.
Some of those cells are over-counted (center inside, but part of the cell
sticks out); some are under-counted (center outside, but part of the cell is in
the disk).

Three facts you'll want when you interpret your own plot:

- **How many cells does the edge cut through?** A curve of length $L$ crosses
  roughly $L/h$ cells, so the number of edge cells grows like $N$: double $N$,
  twice as many edge cells.
- **How big is each one's mistake?** At most about one cell's worth,
  $h^2 \times f$, so each one's mistake shrinks like $1/N^2$.
- **Do the mistakes add up or cancel?** If they all had the same sign, the total
  would be (about $N$ cells) $\times$ (about $1/N^2$ each). But they don't all have
  the same sign. A rule of thumb from statistics: add up $N$ numbers that are
  randomly $+$ or $-$, and the total typically grows like $\sqrt N$, not $N$. And
  *how* well they cancel depends on exactly where the grid lines happen to fall
  on the circle, which changes from one $N$ to the next.

## 9. Your turn: the quarter disk

Copy of `riemann_rectangle_demo` with the mask from `disk_area_demo` added.
Change the marked lines so it does **our** problem:

1. the grid covers the square $[0, 1] \times [0, 1]$ (so what is $h$?);
2. $f = xy$;
3. `inside` should be `True` exactly for points in the quarter disk. (Since the grid
   already has $x \ge 0$ and $y \ge 0$, you only need the $x^2 + y^2 \le 1$ part.)

As handed out, the mask keeps every point, so it's a plain rectangle.

In [ ]:
def my_riemann(N):
    # Change the line below.
    h = (np.pi / 2) / N
    x_mid = (np.arange(N) + 0.5) * h
    y_mid = (np.arange(N) + 0.5) * h
    X, Y = np.meshgrid(x_mid, y_mid, indexing='xy')
    # Change the line below.
    f_vals = np.cos(X) * np.cos(Y)
    # Change the line below.
    inside = X >= 0
    return np.sum(f_vals * inside) * h * h

print('N = 10:', my_riemann(10), '  exact:', float(exact))

Now run every $N$ from 10 to 320 in steps of 5 and plot the absolute error on
log-log axes. The problem's five doubling values, $N = 10, 20, 40, 80, 160$,
are drawn as big red dots; the cell also prints their errors, with the sign,
so you can check your "above or below" prediction.

In [ ]:
N_vals = np.arange(10, 325, 5)
errors = np.zeros(len(N_vals))
for i in range(len(N_vals)):
    errors[i] = abs(my_riemann(N_vals[i]) - float(exact))

N_doubling = np.array([10, 20, 40, 80, 160])
errors_doubling = np.zeros(len(N_doubling))
for i in range(len(N_doubling)):
    signed_error = my_riemann(N_doubling[i]) - float(exact)
    errors_doubling[i] = abs(signed_error)
    print('N =', N_doubling[i], '  Riemann sum - exact =', signed_error)

plt.figure(figsize=(7, 5))
plt.loglog(N_vals, errors, '.-', label='every N from 10 to 320')
plt.loglog(N_doubling, errors_doubling, 'ro', markersize=8, label='N = 10, 20, 40, 80, 160')
add_reference_lines(N_vals, errors)
plt.xlabel('N')
plt.ylabel('absolute error')
plt.legend()
plt.show()

## 10. What you found

**(a)** Do your by-hand, SymPy, and Riemann-sum answers agree? Was your "above
or below" prediction right?

*(your answer here)*

**(b)** What slope do you read off your plot? In two or three sentences, explain why the
observed slope is what it is, rather than the slope $-2$ you got integrating a smooth
function over a plain rectangle in section 6.

*(your answer here)*

**(c)** One thing that went wrong along the way (mismatched `meshgrid` indexing is
the classic; an $h$ that didn't match the square; a mask that kept everything; ...).

*(your answer here)*

## 11. Optional extension (no extra credit, just fun)

Use the same one-dimensional Riemann-sum machinery as section 4 to check the
**Gaussian moment identity**: if $X$ is a zero-mean Gaussian random variable
of width $\sigma$, with probability density

$$p(x) = \frac{1}{\sigma\sqrt{2\pi}} e^{-x^2/2\sigma^2},$$

then the average of $e^{cX}$ is

$$\langle e^{cX} \rangle = \int_{-\infty}^{\infty} e^{cx} p(x)\, dx = e^{c^2\sigma^2/2}.$$

Pick a $c$ and a $\sigma$. You can't sum out to infinity, but $p(x)$ is tiny beyond
about $10\sigma$, so a midpoint sum over $[-10\sigma, 10\sigma]$ will do (the midpoints
start at $-10\sigma + h/2$).

In [ ]:
# optional: your Gaussian check here